# Tutorial - Seals

There are three physics-based seal models available in ROSS:

- `LabyrinthSeal` — multi-tooth compressible throttling seal.

- `HolePatternSeal` — bulk-flow model for annular seals with hole / honeycomb–type cells (pocket damper seals).

- `HybridSeal` — combined hole-pattern + labyrinth with iterative **_mass-flow matching_** at the interface.

When stiffness and damping coefficients are already known use the general class `SealElement` instead. It does not compute coefficients from geometry — you supply them directly.

>**Note**: Seal elements are excluded from static analysis as they do not support the rotor weight.

In [1]:
# Make sure the default renderer is set to 'notebook' for inline plots in Jupyter
import plotly.io as pio

pio.renderers.default = "notebook"

# Section 1: Labyrinth Seal

A labyrinth restricts leakage through a series of throttlings (teeth). ROSS solves compressible flow along the seal, estimates **mass flow** (including choke where relevant), **swirl** in cavities, and **perturbed** equations for **stiffness and damping** (and optional fluid-related mass terms) versus whirl speed.

**Typical inputs:** shaft diameter, radial clearance, number of teeth, pitch and tooth geometry, seal type (`"rotor"`, `"stator"`, or `"inter"`), pressures, temperature, shaft frequency(ies), preswirl, and gas data.

In [2]:
from ross.seals.labyrinth_seal import LabyrinthSeal
from ross.units import Q_

seal = LabyrinthSeal(
    n=0,
    shaft_diameter=Q_(145, "mm"),
    radial_clearance=Q_(0.3, "mm"),
    n_teeth=16,
    pitch=Q_(3.175, "mm"),
    tooth_height=Q_(3.175, "mm"),
    tooth_width=Q_(0.1524, "mm"),
    seal_type="inter",
    inlet_pressure=308_000.0,
    outlet_pressure=94_300.0,
    inlet_temperature=283.15,
    frequency=Q_([5000, 8000, 11000], "RPM"),
    preswirl=0.98,
    gas_composition={"Nitrogen": 0.79, "Oxygen": 0.21},
)
seal

c:\Users\vinic\OneDrive\Desktop\Digital_Twin\Github\ross\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LabyrinthSeal(n=0, n_link=None,
 kxx=[-1.94449649e+04 -3.15819258e+09  2.80261312e+15], kxy=[ 1.65479279e+04 -7.83714166e+09  9.36204948e+15],
 kyx=[-1.65479279e+04  7.83714166e+09 -9.36204948e+15], kyy=[-1.94449649e+04 -3.15819258e+09  2.80261312e+15],
 kzz=[0, 0, 0], cxx=[ 1.46954145e+01 -1.03899648e+07  7.99109118e+12],
 cxy=[ 3.54252215e+01  2.60730549e+06 -2.55043342e+12], cyx=[-3.54252215e+01 -2.60730549e+06  2.55043342e+12],
 cyy=[ 1.46954145e+01 -1.03899648e+07  7.99109118e+12], czz=[0, 0, 0],
 mxx=[0, 0, 0], mxy=[0, 0, 0],
 myx=[0, 0, 0], myy=[0, 0, 0],
 mzz=[0, 0, 0],
 frequency=[ 523.5987756   837.75804096 1151.91730632], tag=None)

**Access:** Coefficients are stored as lists aligned with `frequency` (for example `seal.kxx`, `seal.kxy`, …; `seal.cxx`, …). Leakage: `seal.seal_leakage` — total mass flow rate in **kg/s**, one value per entry in `frequency`.

**Without** `gas_composition`, you must supply **`molar_mass`**, **`gamma`**, **`reference_temperatures`** (two temperatures [K]), and **`reference_viscosities`** (two dynamic viscosities [Pa·s]).

# Section 2: Hole-pattern Seal

Hole-pattern (including **honeycomb** stator) seals are represented by a **one-dimensional bulk-flow** model along the seal length: base compressible flow, perturbation for dynamic coefficients, integrated **direct and cross** stiffness, damping, and apparent mass terms.  

Geometrically the model uses a **representative cell** (`cell_length`, `cell_width`, `cell_depth`) and **relative roughness** (dimensionless `E/D`), not a full CAD mesh.

In [3]:
from ross.seals.holepattern_seal import HolePatternSeal
from ross.units import Q_

# Geometry derived from honeycomb_data.pdf (see table above)
DRUM_OD_M = 196.5e-3
shaft_diameter = DRUM_OD_M
length_honeycomb = 84.9e-3
diametral_clearance = 0.36e-3
radial_clearance = diametral_clearance / 2

holepattern = HolePatternSeal(
    n=0,
    shaft_diameter=shaft_diameter,
    radial_clearance=radial_clearance,
    axial_length=length_honeycomb,
    relative_roughness=1.0e-4,
    cell_length=length_honeycomb / 37,
    cell_width=2.1e-3,
    cell_depth=2.8e-3,
    inlet_pressure=1_830_000.0,
    outlet_pressure=823_500.0,
    inlet_temperature=300.0,
    frequency=Q_([5000], "RPM"),
    gas_composition={
        "Nitrogen": 0.7812,
        "Oxygen": 0.2096,
        "Argon": 0.0092,
    },
    preswirl=1.0,
    entrance_loss_coefficient=0.1,
    exit_loss_coefficient=0.5,
    nz=40,
)
holepattern

HolePatternSeal(n=0, n_link=None,
 kxx=[5760043.48572718], kxy=[1377442.42449097],
 kyx=[-1377442.42449097], kyy=[5760043.48572718],
 kzz=[0], cxx=[2470.55767601],
 cxy=[-477.40987914], cyx=[477.40987914],
 cyy=[2470.55767601], czz=[0],
 mxx=[-0.4221625], mxy=[0.11170356],
 myx=[-0.11170356], myy=[-0.4221625],
 mzz=[0],
 frequency=[523.5987756], tag=None)

# Section 3: Hybrid Seal

This tutorial demonstrates how to use the `HybridSeal` class to model a hybrid seal that combines a hole-pattern seal (damping section) with a labyrinth seal (throttling section). The hybrid seal model iteratively determines the interface pressure between the two seal stages by matching their leakage rates, then combines their rotordynamic coefficients.

The hybrid seal configuration in ROSS consists of:

1. **Hole-Pattern Seal (Damping Stage)**: Located upstream, this section provides significant direct damping to improve rotor stability. The honeycomb-like surface increases energy dissipation due to recirculation and cavity pressurization effects, producing high damping.

2. **Labyrinth Seal (Throttling Stage)**: Located downstream, this section provides the primary pressure drop through a series of throttling cavities (teeth). The labyrinth geometry creates a tortuous flow path that minimizes leakage.

## 3.1 Iterative Pressure Matching

The model uses a bisection method to find the interface pressure ($P_{int}$) between the two seal stages that satisfies mass conservation:

$$\dot{m}_{hole-pattern}(P_{in}, P_{int}) = \dot{m}_{labyrinth}(P_{int}, P_{out})$$

where:
- $P_{in}$ is the inlet pressure (upstream of hole-pattern seal)
- $P_{out}$ is the outlet pressure (downstream of labyrinth seal)
- $\dot{m}$ represents the mass flow rate (leakage)

## 3.2 Hybrid Seal Parameters

The `HybridSeal` requires parameters for both the hole-pattern and labyrinth seal sections, as well as common operating conditions.

### 3.2.1 Common Parameters

These parameters are shared between both seal stages:

| Parameter | Description | Unit |
|-----------|-------------|------|
| `n` | Node location in the rotor model | - |
| `shaft_diameter` | Diameter of the shaft | m |
| `inlet_pressure` | Total inlet pressure at hole-pattern entrance | Pa |
| `outlet_pressure` | Final outlet pressure at labyrinth exit | Pa |
| `inlet_temperature` | Inlet temperature | K |
| `frequency` | Shaft rotational speed(s) | rad/s |
| `gas_composition` | Gas composition dictionary (optional) | - |
| `molar_mass` | Molecular mass (required if no gas_composition) | kg/kgmol |
| `gamma` | Specific heat ratio Cp/Cv (required if no gas_composition) | - |

In [ ]:
from ross import HybridSeal
from ross.units import Q_

# Common parameters for the hybrid seal
n = 3  # Node location
shaft_diameter = Q_(50, "mm")  # 50 mm shaft diameter
inlet_pressure = 500000  # 5 bar inlet pressure (Pa)
outlet_pressure = 100000  # 1 bar outlet pressure (Pa)
inlet_temperature = 300.0  # 300 K (~27°C)
frequency = Q_([2000, 3000, 4000, 5000], "RPM")  # Operating speeds

# Gas composition for air
gas_composition = {
    "Nitrogen": 0.7812,
    "Oxygen": 0.2096,
    "Argon": 0.0092,
}

### 3.2.2 Hole-Pattern Seal Parameters

The hole-pattern seal (damping stage) parameters define the geometry of the porous surface:

| Parameter | Description | Unit |
|-----------|-------------|------|
| `radial_clearance` | Seal clearance | m |
| `axial_length` | Length of the seal | m |
| `relative_roughness` | Surface roughness ratio (ε/D) | - |
| `cell_length` | Axial length of each cell | m |
| `cell_width` | Circumferential width of each cell | m |
| `cell_depth` | Depth of each cell | m |
| `preswirl` | Inlet swirl ratio | - |
| `entrance_loss_coefficient` | Entrance loss coefficient | - |
| `exit_loss_coefficient` | Exit loss coefficient | - |

In [ ]:
hole_pattern_params = {
    "radial_clearance": 0.0003,  # 0.3 mm clearance
    "axial_length": 0.04,  # 40 mm length
    "relative_roughness": 0.0001,  # Relative roughness
    "cell_length": 0.003,  # 3 mm cell length
    "cell_width": 0.003,  # 3 mm cell width
    "cell_depth": 0.002,  # 2 mm cell depth
    "preswirl": 0.8,  # 80% preswirl
    "entrance_loss_coefficient": 0.5,  # Entrance loss coefficient
    "exit_loss_coefficient": 1.0,  # Exit loss coefficient
}

### 3.2.3 Labyrinth Seal Parameters

The labyrinth seal (throttling stage) parameters define the tooth geometry:

| Parameter | Description | Unit |
|-----------|-------------|------|
| `radial_clearance` | Nominal radial clearance | m |
| `n_teeth` | Number of teeth (throttlings) | - |
| `pitch` | Axial cavity length (land length) | m |
| `tooth_height` | Height of seal strip | m |
| `tooth_width` | Thickness of throttle (tip-width) | m |
| `seal_type` | Location of teeth: 'rotor', 'stator', or 'inter' | - |
| `preswirl` | Inlet swirl velocity ratio | - |
| `reference_temperatures` | Temperature at states [T1, T2] (required if no gas_composition) | K |
| `reference_viscosities` | Dynamic viscosity at states [μ1, μ2] (required if no gas_composition) | kg/(m·s) |


In [ ]:
labyrinth_params = {
    "radial_clearance": Q_(0.25, "mm"),  # 0.25 mm clearance
    "n_teeth": 10,  # 10 teeth
    "pitch": Q_(3, "mm"),  # 3 mm pitch
    "tooth_height": Q_(3, "mm"),  # 3 mm tooth height
    "tooth_width": Q_(0.15, "mm"),  # 0.15 mm tooth width
    "seal_type": "inter",  # Interlocking type
    "preswirl": 0.9,  # 90% preswirl
    "reference_temperatures": [300.0, 299.5],  # Temperature at states (K)
    "reference_viscosities": [1.85e-05, 1.84e-05],  # Dynamic viscosity (kg/(m·s))
}

## 3.3 Creating the Hybrid Seal

Now we can create the `HybridSeal` object:

In [ ]:
hybrid_seal = HybridSeal(
    n=n,
    shaft_diameter=shaft_diameter,
    inlet_pressure=inlet_pressure,
    outlet_pressure=outlet_pressure,
    inlet_temperature=inlet_temperature,
    frequency=frequency,
    gas_composition=gas_composition,
    hole_pattern_parameters=hole_pattern_params,
    labyrinth_parameters=labyrinth_params,
)

## 3.4 Analyzing the Results

### 3.4.1 Summary Results

The `summary_results()` method provides a quick overview of the hybrid seal analysis:


In [ ]:
hybrid_seal.summary_results()

### 3.4.2 Rotordynamic Coefficients

The `format_table()` method displays the frequency-dependent rotordynamic coefficients:


In [ ]:
hybrid_seal.format_table(frequency_units="RPM")

### 3.4.3 Convergence Analysis

The `plot_convergence()` method shows how the iterative solution converges to the final interface pressure. The convergence plot shows:
1. **Convergence History**: The relative leakage error decreasing with each iteration

2. **Leakage Rate Convergence**: How the leakage rates from both seals approach each other

3. **Interface Pressure Evolution**: The bisection method narrowing down to the correct interface pressure


In [ ]:
fig_conv = hybrid_seal.plot_convergence()
fig_conv.show()

### 3.4.4 Pressure Distribution

The `plot_pressure_distribution()` method shows the pressure profile through both seal stages:


In [ ]:
fig_pressure = hybrid_seal.plot_pressure_distribution(pressure_units="bar")
fig_pressure.show()

## 3.5 Alternative Configuration


If CoolProp is not available or you prefer to specify gas properties manually, you can omit the `gas_composition` parameter and provide `molar_mass` and `gamma` instead. In this case, you also need to provide viscosity model parameters for the hole-pattern seal (`sutherland_b`, `sutherland_s`).

In [ ]:
# Hole-pattern parameters with manual viscosity model
hole_pattern_params_manual = {
    "radial_clearance": 0.0003,
    "axial_length": 0.04,
    "relative_roughness": 0.0001,
    "cell_length": 0.003,
    "cell_width": 0.003,
    "cell_depth": 0.002,
    "preswirl": 0.8,
    "entrance_loss_coefficient": 0.5,
    "exit_loss_coefficient": 1.0,
    "sutherland_b": 1.458e-6,  # Sutherland viscosity coefficient b
    "sutherland_s": 110.4,  # Sutherland viscosity coefficient s
}

# Create hybrid seal with manual gas properties
hybrid_seal_manual = HybridSeal(
    n=3,
    shaft_diameter=Q_(50, "mm"),
    inlet_pressure=500000,
    outlet_pressure=100000,
    inlet_temperature=300.0,
    frequency=Q_([3000], "RPM"),
    molar_mass=28.97,  # Air molecular mass (kg/kgmol)
    gamma=1.4,  # Air specific heat ratio
    hole_pattern_parameters=hole_pattern_params_manual,
    labyrinth_parameters=labyrinth_params,
)

hybrid_seal_manual.summary_results()